# 07 — Train a Mini-Former

**Topic:** Train a mini-former  
**Audience:** AI PMs, founders, ML engineers, and portfolio builders.  
**How to use this notebook:** run the toy implementation first, then replace the toy data/model with your company data or project data.

## Mental model

This notebook is part of a 32-part LLM systems implementation path. Each notebook follows the same pattern:

1. Product intuition: what capability this unlocks.
2. Core idea: the minimum math or architecture you need.
3. Implementation from scratch or near-scratch.
4. Production notes: cost, risk, reliability, and founder decisions.
5. Portfolio extension: how to turn this into a public project.


In [ ]:
from pathlib import Path
import math, random, re, json, os, statistics, time, queue, threading
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except Exception as e:
    TfidfVectorizer = None
    cosine_similarity = None
    print('Optional sklearn import failed:', e)

# Make notebooks runnable from the pack root OR from a notebook subdirectory.
def find_pack_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for p in [start] + list(start.parents):
        if (p / 'data' / 'tiny_corpus.txt').exists():
            return p
    # Fallback for the sandbox path used to create this pack.
    sandbox = Path('/mnt/data/advanced_llm_systems_notebook_pack')
    if (sandbox / 'data' / 'tiny_corpus.txt').exists():
        return sandbox
    return start

ROOT = find_pack_root()
DATA = ROOT / 'data'
print('Pack root:', ROOT)
print('PyTorch:', torch.__version__)
torch.set_num_threads(1)
torch.manual_seed(7)
random.seed(7)
np.random.seed(7)

def basic_tokenize(text: str):
    return re.findall(r"[a-zA-Z]+|\d+|[^\w\s]", text.lower())

def count_params(model):
    return sum(p.numel() for p in model.parameters())

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B,T,C = x.shape
        q,k,v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k = k.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        v = v.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        att = q @ k.transpose(-2,-1) / math.sqrt(self.d_head)
        mask = torch.tril(torch.ones(T,T,device=x.device)).bool()
        att = att.masked_fill(~mask[None,None,:,:], float('-inf'))
        w = torch.softmax(att, dim=-1)
        y = self.dropout(w) @ v
        y = y.transpose(1,2).contiguous().view(B,T,C)
        return self.proj(y)

class TransformerBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4, mlp_ratio=4, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio*d_model), nn.GELU(),
            nn.Linear(mlp_ratio*d_model, d_model), nn.Dropout(dropout)
        )
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

# Shared toy RAG utilities for notebooks that need retrieval.
def load_rag_docs():
    path = DATA / 'rag_docs.jsonl'
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

docs = load_rag_docs()
_retriever_state = {}
def build_retriever(docs_in=None):
    docs_in = docs if docs_in is None else docs_in
    if not docs_in or TfidfVectorizer is None:
        return None, None
    vectorizer = TfidfVectorizer(stop_words='english')
    X = vectorizer.fit_transform([d['title'] + ' ' + d['text'] for d in docs_in])
    return vectorizer, X

_default_vectorizer, _default_X = build_retriever(docs)
def retrieve(query, k=2):
    if _default_vectorizer is None or _default_X is None:
        return []
    q = _default_vectorizer.transform([query])
    sims = cosine_similarity(q, _default_X)[0]
    idx = sims.argsort()[::-1][:k]
    return [{**docs[i], 'score': float(sims[i])} for i in idx]

# Shared eval helpers.
def exact_contains(output, expected):
    return str(expected).lower() in str(output).lower()

def valid_json_with_key(output, key):
    try:
        return key in json.loads(output)
    except Exception:
        return False

def run_eval(cases, system_fn):
    rows = []
    for c in cases:
        out = system_fn(c['input'])
        if 'expected' in c:
            passed = exact_contains(out, c['expected'])
        elif 'expected_key' in c:
            passed = valid_json_with_key(out, c['expected_key'])
        else:
            passed = False
        rows.append({**c, 'output': out, 'passed': passed})
    return pd.DataFrame(rows)


## Why this matters

Training a tiny language model gives you intuition for data, objectives, loss curves, sampling, overfitting, and compute budgets.

This notebook trains a character-level GPT-like model on a tiny corpus. It is not intended to produce good text; it is intended to make the mechanics transparent.


In [ ]:
text = (DATA / 'tiny_corpus.txt').read_text().lower()
chars = sorted(set(text))
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
vocab_size = len(chars)
print('chars:', ''.join(chars))
print('vocab_size:', vocab_size, 'tokens:', len(data))


In [ ]:
block_size = 32
batch_size = 16

def get_batch(split='train'):
    n = int(0.9 * len(data))
    source = data[:n] if split == 'train' else data[n-block_size:]
    ix = torch.randint(0, len(source)-block_size-1, (batch_size,))
    x = torch.stack([source[i:i+block_size] for i in ix])
    y = torch.stack([source[i+1:i+block_size+1] for i in ix])
    return x, y


In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2, block_size=32):
        super().__init__()
        self.block_size = block_size
        self.tok = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Embedding(block_size, d_model)
        self.blocks = nn.Sequential(*[TransformerBlock(d_model, n_heads, dropout=0.1) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
    def forward(self, idx, targets=None):
        B,T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok(idx) + self.pos(pos)[None,:,:]
        x = self.blocks(x)
        logits = self.head(self.ln(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B*T, -1), targets.reshape(B*T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=80, temperature=0.9):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

model = TinyGPT(vocab_size)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
print('params:', count_params(model))


In [ ]:
losses = []
for step in range(120):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(float(loss))
    if step % 30 == 0:
        print(step, float(loss))


In [ ]:
plt.figure(figsize=(5,3))
plt.plot(losses)
plt.title('Mini-former training loss')
plt.xlabel('step')
plt.ylabel('cross entropy')
plt.show()

start = torch.tensor([[stoi['a']]], dtype=torch.long)
sample = model.generate(start, max_new_tokens=160, temperature=0.8)[0].tolist()
print(''.join(itos[i] for i in sample))


## Founder notes

Tiny training runs are great for portfolio and debugging. Production pretraining is a different game: data quality, distributed systems, hardware budgets, checkpointing, monitoring, and evals dominate.


## Portfolio / company reuse checklist

To reuse this notebook in a real project:

- Replace the toy corpus/data with a small but representative sample from your domain.
- Add a held-out evaluation set before optimizing anything.
- Track failure cases by category, not just aggregate accuracy/loss.
- Decide whether this is a prototype-only component, a production component, or a learning artifact.
- Document assumptions in a model card or system card.


## References to review after implementation

Use the implementation above first, then read the original papers/docs to connect the code to production systems. The README contains a consolidated reference list for the full pack.
